# Argumentation strate 6 — Acceptabilité QBF : quantificateurs par énumération, crédule et sceptique

**Série** : ICT-Series (strate 6, argumentation)

Ce carnet est le troisième de la famille `ICT-Argumentation-*` : après les trajectoires de croyance Dung ([`ICT-Argumentation-BeliefTrajectories-Python`](ICT-Argumentation-BeliefTrajectories-Python.ipynb)) et la mémoire de justification JTMS/ATMS (`ICT-Argumentation-TruthMaintenance-Python`), il ajoute l'angle **complexité computationnelle** : lire l'*acceptabilité* d'un argument comme une **formule booléenne quantifiée** (QBF) — et la résoudre à la main, par énumération naïve, pour voir ce que « quantifier » veut dire.

**Cadre.** Une QBF est une formule propositionnelle dont les variables sont préfixées de quantificateurs ∀ (pour tout) et ∃ (il existe). Décider si une QBF est valide est le problème canonique **PSPACE-complet** (Papadimitriou 1994) : chaque bloc de quantificateurs multiplie le travail de l'évaluateur. Notre moteur n'est PAS un solveur SOTA — c'est le contraire assumé : une **énumération exhaustive** en `2^n` qui rend chaque quantificateur visible comme une boucle. C'est précisément la leçon : le quantificateur n'est pas une notation, c'est un coût.

**Statut épistémique.** Ce carnat est une *restitution pédagogique mesurée* : les sorties committées proviennent d'un port dont la fidélité à la source est prouvée par **exécution différentielle** (16/16 surfaces identiques, tête `a5ac1a5d`) et par des littéraux figés dans [`ict/tests/test_qbf.py`](ict/tests/test_qbf.py) (21 gates). Aucun chiffre de ce carnet n'est écrit à la main : chaque valeur citée dans la prose est lue sur la sortie committée de la cellule qui la produit.

**Organ-first — copie pédagogique déclarée.** Le moteur [`ict/qbf.py`](ict/qbf.py) est un port déclaré du module EPITA `argumentation_analysis/agents/core/logic/qbf_native.py` (né au commit `0d2ac1b0`), enseigné par le notebook EPITA `belief_revision.ipynb` §5. Aucun moteur QBF n'existe sous forme d'organe invoquable dans ce dépôt : la seule sémantique Dung réelle vit dans [`ict/argumentation.py`](ict/argumentation.py), que ce carnet **étend** d'une fonction `preferred_extensions` (l'organe calcule désormais la sémantique dont la question sceptique a besoin). Les détails vivent dans la section conclusion.

## 1. Setup — les organes

Deux organes : [`ict.argumentation`](ict/argumentation.py) (cadres de Dung, labeling grounded — et depuis ce carnet, extensions préférées) et [`ict.qbf`](ict/qbf.py) (AST à quantificateurs, parseur, énumérateur, acceptabilité crédule/sceptique). Convention de la série : numpy-free, déterminisme complet, CPU pur. Les sorties n'impriment que des `basename` — aucun chemin machine.

In [1]:
import os
import sys

ICT_ROOT = os.path.abspath('.')
if ICT_ROOT not in sys.path:
    sys.path.insert(0, ICT_ROOT)

import ict
from ict import argumentation as arg
from ict import qbf

print('organe argumentation :', os.path.basename(arg.__file__))
print('organe qbf           :', os.path.basename(qbf.__file__))
print('semantiques exposees : grounded_labeling, est_admissible, preferred_extensions')

organe argumentation : argumentation.py
organe qbf           : qbf.py
semantiques exposees : grounded_labeling, est_admissible, preferred_extensions


## 2. Quantificateurs : ce que « pour tout » et « il existe » coûtent

### 2.1 Les deux cas extrêmes

La tautologie et la contradiction sont les bornes : la première valide **quelle que soit** la valeur de `x`, la seconde invalide **quelle que soit** la recherche d'un témoin. Remarquez le vocabulaire du verdict : `VALID` / `INVALID` — le moteur parle le langage des QBF, pas celui des ensembles de modèles.

In [2]:
# forall x. (x | !x) : tautologie -- valide pour toute valeur.
print(qbf.check_qbf([{"type": "forall", "vars": ["x"]}], "x | !x"))

# exists x. (x & !x) : contradiction -- aucun temoin n'existe.
print(qbf.check_qbf([{"type": "exists", "vars": ["x"]}], "x & !x"))

(True, 'QBF VALID: x | !x')
(False, 'QBF INVALID: x & !x')


### 2.2 L'alternance — et pourquoi l'ordre n'est pas commutatif

Le cœur du sujet n'est pas chaque quantificateur isolé mais leur **alternance**. Prenons la matrice d'équivalence `x & y | !x & !y` (lue : `x == y`). Les deux préfixes possibles à deux blocs rendent des verdicts **opposés** sur la même matrice :

- `exists x. forall y. (x == y)` — **invalide** : on choisit `x` d'abord, puis l'adversaire (`forall`) choisit `y` pour démentir. Un seul choix ne peut pas couvrir deux valeurs.
- `forall y. exists x. (x == y)` — **valide** : l'adversaire bouge d'abord, et le répondant copie. Répondre après est plus fort que jouer avant.

Cette lecture **stratégique** (le joueur existentiel contre le joueur universel) est exactement la sémantique des jeux des QBF : chaque préfixe est un ord de jeu. L'énumération naïve la matérialise — boucle imbriquée après boucle imbriquée.

In [3]:
matrix = "x & y | !x & !y"  # x == y

ex_for = qbf.check_qbf([{"type": "exists", "vars": ["x"]},
                        {"type": "forall", "vars": ["y"]}], matrix)
for_ex = qbf.check_qbf([{"type": "forall", "vars": ["y"]},
                        {"type": "exists", "vars": ["x"]}], matrix)
print("exists x. forall y . (x == y) ->", ex_for[0])
print("forall y. exists x . (x == y) ->", for_ex[0])
assert ex_for[0] is False and for_ex[0] is True

exists x. forall y . (x == y) -> False
forall y. exists x . (x == y) -> True


### 2.3 Trois blocs : le répondant intérieur

À trois blocs, la même logique se poursuit un cran plus loin. Sur la matrice à trois clauses `a & b | !b & c | !a & !c`, le préfixe `exists a. forall b. exists c.` est **valide** : le premier joueur pose `a = Vrai`, l'adversaire choisit `b`, et le répondant intérieur `c` **répond à `b`** (`c = Vrai` si `b = Faux`). Le quantificateur le plus interne joue en dernier — c'est lui qui dispose de l'information de tous les coups précédents.

In [4]:
triple = qbf.check_qbf(
    [{"type": "exists", "vars": ["a"]},
     {"type": "forall", "vars": ["b"]},
     {"type": "exists", "vars": ["c"]}],
    "a & b | !b & c | !a & !c")
print(triple[0], '-', triple[1])

# Le cout rendu visible : analyze_qbf rapporte l'espace de recherche.
r = qbf.analyze_qbf(
    [{"type": "forall", "vars": ["x", "y"]}, {"type": "exists", "vars": ["z"]}],
    "x & y & z")
print('search_space =', r['statistics']['search_space'],
      '| variables =', r['statistics']['variable_count'],
      '| blocs =', r['statistics']['quantifier_count'])

True - QBF VALID: a & b | !b & c | !a & !c
search_space = 8 | variables = 3 | blocs = 2


La statistique `search_space = 8` est le **prix affiché** : trois variables, `2**3 = 8` feuilles au pire cas. C'est là que la leçon de complexité se tient : ajouter une variable **double** l'espace ; ajouter un bloc quantifié en déplace la structure. Un solveur SOTA (résolution, expansion de préfixe) fait mieux que l'énumération — mais l'énumération est la seule méthode qui montre le problème **en entier**, feuille par feuille, raison pour laquelle ce moteur pédagogique l'assume et la déclare.

Cette grille de lecture stratégique se réutilise telle quelle en théorie des jeux : un préfixe QBF est un jeu à information parfaite où les joueurs posent leurs variables à tour de rôle, le joueur ∃ cherchant à rendre la matrice vraie, le joueur ∀ à la rendre fausse. La QBF est valide exactement quand le joueur ∃ a une **stratégie gagnante** — une fonction qui, à chaque coup adverse, associe une réponse correcte. C'est pourquoi l'ordre compte : une stratégie doit répondre au coup précédent, pas le deviner. Le répondant intérieur d'une alternance `∃∀∃` voit les coups de ses deux adversaires successifs ; le quantificateur externe, lui, joue à l'aveugle. Toute la hiérarchie polynomiale (∃, ∀, ∃∀, ∀∃...) se relit ainsi : une échelle d'information décroissante pour le joueur existentiel.

### Exercice 1 — un ordre de quantificateurs qui change le verdict

**Contexte.** La section 2.2 a montré que `forall y. exists x. (x == y)` est valide et que `exists x. forall y. (x == y)` est invalide. **Objectif.** Construire une autre matrice à deux variables, différente de l'équivalence, pour laquelle `forall y. exists x.` est **valide** et `exists x. forall y.` est **invalide**. Appelez `qbf.check_qbf` avec les deux préfixes, comme dans la section 2.2.

**Attention.** L'implication `exists x. forall y. M` ⟹ `forall y. exists x. M` est un théorème : le répondant peut toujours suivre le choix de l'adversaire. Ne cherchez donc pas la direction contraire.

**Indice.** Cherchez une matrice où, pour chaque valeur de `y`, il existe un `x` qui la rend vraie, mais où aucun `x` unique ne la rend vraie quelle que soit la valeur de `y`. Les opérateurs `!`, `&` et `|` suffisent.

**Critère de vérification.** Deux verdicts opposés affichés par `check_qbf`, et une phrase qui explique pourquoi le répondant intérieur gagne dans un ordre et pas dans l'autre.


In [5]:
# Exercice 1 -- matrice ou l'ordre des quantificateurs decide
# TODO etudiant
def exercice_1_ordre_sensible():
    print("Exercice 1 a completer")
    return None


exercice_1_ordre_sensible()


Exercice 1 a completer


## 3. Le parseur : précédences, et une limite assumée

La formule arrive en **chaîne** (`"x => y"`), pas en AST. Le parseur applique les précédences `! > & > | > =>` — l'implication est la plus faible. **Limite héritée de la source et enseignée ici** : pas de parenthèses. Le `repr` de l'AST montre la structure retenue, et avec elle le piège : `a & b | c` se lit `(a & b) | c`, jamais `a & (b | c)`.

In [6]:
f = qbf.parse_formula("a & b | c")
print(type(f).__name__, '->', repr(f))

g = qbf.parse_formula("a => b & c")
print(type(g).__name__, '->', repr(g))
# => est la plus faible : a => (b & c), jamais (a => b) & c.
assert repr(g) == '(a => (b & c))'

Or -> ((a & b) | c)
Implies -> (a => (b & c))


Ce choix n'est pas un bug silencieux : il est documenté dans la docstring du module et épingle par un test (`test_parseur_precedences_sans_parentheses`). En pédagogie, une limite **montrée** vaut mieux qu'une généralité cachée : l'étudiant voit qu'un parseur sans parenthèses n'est pas neutre — il *choisit* une lecture pour vous.

## 4. Acceptabilité crédule : l'énumération EST le quantificateur

Revenons à l'argumentation. La question **crédule** est : *existe-t-il* une extension admissible qui contienne l'argument `target` ? Relisez-la : c'est un **∃** — il existe un ensemble, sans conflit, qui se défend, contenant la cible. La fonction `credulous_acceptance_qbf` ne convertit pas cette question en QBF pour la déléguer à un solveur : elle **matérialise** le ∃ en énumérant les sous-ensembles (`2**n` masques binaires croissants) et en s'arrêtant au **premier témoin**. La recherche de témoin et le quantificateur existentiel sont le même objet vu par la théorie et par le code.

Sur le *Nixon diamond* à trois arguments (a et b s'attaquent, c est libre), chaque argument est crédulement accepté — mais les témoins racontent des choses différentes :

In [7]:
ARGS = ["a", "b", "c"]
ATTACKS = [["a", "b"], ["b", "a"]]

for tgt in ARGS:
    r = qbf.credulous_acceptance_qbf(ARGS, ATTACKS, tgt)
    print(f"cible {tgt} : accepte={r['accepted']} temoin={r.get('witness_extension')}")

# Le temoin est DETERMINISTE : premier masque binaire admissible contenant la cible.
assert qbf.credulous_acceptance_qbf(ARGS, ATTACKS, "a")["witness_extension"] == ["a"]

cible a : accepte=True temoin=['a']
cible b : accepte=True temoin=['b']
cible c : accepte=True temoin=['c']


Deux cas limites mesurés, qui distinguent la sémantique d'une intuition vite faite :

- **La chaîne défensive** — `d` est attaqué par `b`, lui-même attaqué par `a` et `c`. Intuitivement « d est attaqué, donc refusé » ; la mesure dit le contraire : `{a, d}` est admissible (`a` **défend** `d`), témoin `['a', 'd']`. L'acceptabilité crédule est une question de **coalition défensive**, pas de statut individuel.
- **Le cycle impair** — `a -> b -> c -> a` : personne n'est crédulement accepté. Chaque attaquant d'un membre du cycle est un autre membre... qui ne peut pas coexister avec lui (conflit). Le cycle pair (Nixon) laisse deux témoins singleton ; le cycle impair n'en laisse **aucun**.

In [8]:
# Chaine defensive : le temoin contient DEUX arguments (a defend d).
r = qbf.credulous_acceptance_qbf(
    ["a", "b", "c", "d"], [["a", "b"], ["c", "b"], ["b", "d"]], "d")
print('chaine, cible d :', r['accepted'], r['witness_extension'])

# Cycle impair : refuse pour TOUTE cible.
odd = qbf.credulous_acceptance_qbf(
    ["a", "b", "c"], [["a", "b"], ["b", "c"], ["c", "a"]], "a")
print('cycle impair, cible a :', odd['accepted'], '-', odd['reason'])

chaine, cible d : True ['a', 'd']
cycle impair, cible a : False - No admissible extension contains a


> **Encadré — la parité mesurée est un théorème nommé.** La dichotomie mesurée ci-dessus — le cycle pair (Nixon) laisse des témoins, le cycle impair `a → b → c → a` n'en laisse **aucun** — est l'obstruction de parité que la théorie des graphes a nommée : un graphe est **biparti** (ses sommets se répartissent en deux ensembles sans arête interne) si et seulement s'il ne contient **aucun cycle impair** (König). Répartir les arguments en « deux camps sans attaque interne » est exactement une 2-coloration du graphe d'attaque : le théorème dit quand cette répartition existe, et le 3-cycle en est le contre-exemple minimal — la même ligne de parité qui sépare les deux comportements mesurés par l'organe. C'est l'exemple type du principe « généraliser allège » (S. Sheydvasser, *The Power of Abstraction*, 26/09/2026).

La borne `n <= 15` n'est pas une coquetterie : à 16 arguments, `2**16 = 65536` sous-ensembles — l'organe **refuse de mesurer** (`accepted=None`, raison explicite) plutôt que de tourner en silence. Un refus visible vaut mieux qu'un temps de calcul caché : c'est la même discipline que le `RefusDeMesure` du pipeline discursif de la série.

### Exercice 2 — prédire le témoin de l'énumération

**Contexte.** La section 4 a montré que `credulous_acceptance_qbf` renvoie le **premier** masque binaire admissible contenant la cible, les bits étant lus dans l'ordre de la liste `arguments`. **Objectif.** Construire un cadre de Dung à 4 arguments où la cible est crédulement acceptée, mais où tous les témoins admissibles ont **deux** membres (la cible et son défenseur). Prédire le témoin sur papier, puis le vérifier avec `credulous_acceptance_qbf`.

**Indice.** Listez les attaquants de la cible. Une extension sans conflit qui contient la cible doit aussi contenir, pour chacun de ces attaquants, un argument qui l'attaque à son tour. Écrivez l'ordre des masques croissants avant de lancer le calcul.

**Critère de vérification.** La valeur de `witness_extension` est égale à votre prédiction. Le témoin est une liste triée (`sorted`), donc son ordre est alphabétique.


In [9]:
# Exercice 2 -- prediction du temoin
# TODO etudiant
def exercice_2_temoin():
    print("Exercice 2 a completer")
    return None


exercice_2_temoin()


Exercice 2 a completer


## 5. Acceptabilité sceptique : le ∀ passe par l'organe natif

La question **sceptique** inverse le quantificateur : `target` est-il dans **toutes** les extensions préférées (maximales pour l'inclusion) ? C'est un **∀** sur les extensions. Ici le port **consomme l'organe** : la fonction `preferred_extensions` que ce carnet ajoute à [`ict/argumentation.py`](ict/argumentation.py) énumère les ensembles admissiaux maximaux — le mapping noms→indices vit à la frontière, la sémantique vit dans l'organe.

Sur le Nixon à trois arguments, les extensions préférées sont `{a, c}` et `{b, c}` (les singletons `{a}` et `{b}` ne sont pas maximaux : `c` — non attaqué — les complète tous les deux) :

In [10]:
ra = qbf.skeptical_acceptance_qbf(ARGS, ATTACKS, "a")
rc = qbf.skeptical_acceptance_qbf(ARGS, ATTACKS, "c")
print('preferred :', sorted(ra['preferred_extensions']))
print('a sceptiquement accepte :', ra['accepted'], '-', ra['reason'])
print('c sceptiquement accepte :', rc['accepted'])
assert ra['accepted'] is False and rc['accepted'] is True

preferred : [['a', 'c'], ['b', 'c']]
a sceptiquement accepte : False - a is NOT in all preferred extensions
c sceptiquement accepte : True


L'exemple *Tweety* (a et b se contredisent, c attaque b) resserre la lecture : la seule extension préférée est `{a, c}` — `a` est cette fois **sceptiquement** accepté, parce qu'aucne extension préférée ne peut exclure son défenseur structurel. Crédule et sceptique ne sont pas deux forces d'une même question : ce sont **deux quantificateurs** sur deux collections différentes (∃ sur les admissibles, ∀ sur les préférées).

In [11]:
rt = qbf.skeptical_acceptance_qbf(
    ["a", "b", "c"], [["a", "b"], ["b", "a"], ["c", "b"]], "a")
print('tweety preferred :', sorted(rt['preferred_extensions']))
print('a sceptiquement accepte :', rt['accepted'])
assert rt['accepted'] is True

tweety preferred : [['a', 'c']]
a sceptiquement accepte : True


Récapitulons la dualité telle que le code la montre, colonne par colonne : la question crédule parcourt les **admissibles** (ensembles sans conflit qui se défendent — sans exigence de maximalité) et s'arrête au premier témoin ; la question sceptique parcourt les **préférées** (les admissibles maximaux, obtenus par l'organe) et exige la présence dans toutes. Entre les deux collections, la maximalité : elle ne change aucun statut sur les cas symétriques (Nixon), mais elle change tout dès qu'un argument non attaqué vient compléter les mondes possibles — `c` rejoint chaque préférée du Nixon sans jamais être mis en cause. L'agent crédule et l'agent sceptique divergent exactement là où la structure rend un monde inévitable pour l'un et contingent pour l'autre.

## 6. Le pont : théorème de Dung relu comme inclusion, et témoin négatif

Il reste à relier les deux organes du carnet précédent et celui-ci. Le théorème structurel (Dung 1995) : **l'extension grounded est incluse dans chaque extension préférée**. Le grounded — construit par point fixe du labeling (voir [`ICT-Argumentation-BeliefTrajectories-Python`](ICT-Argumentation-BeliefTrajectories-Python.ipynb)) — est le noyau minimal ; chaque préférée est un monde maximal cohérent avec ce noyau. En langage quantifié : ce qui est accepté *sans hypothèse* (∀ implicite du point fixe) survit dans *tous* les mondes maximaux.

Le bloc suivant est le **témoin négatif** du carnet : il échoue si l'organe Dung dérive, si `preferred_extensions` diverge de la sémantique, ou si le labeling grounded change. Il rejoue l'inclusion sur quatre instances à chaque exécution.

In [12]:
cas = [
    ("nixon 2", arg.nixon_diamond()),
    ("tweety", arg.tweety_bird()),
    ("chaine a->b", arg.DungAF([0, 1], [(0, 1)])),
    ("double defense", arg.DungAF([0, 1, 2], [(0, 1), (2, 1), (1, 2)])),
]
for nom, af in cas:
    lab = arg.grounded_labeling(af)
    grounded_in = {a for a, s in lab.items() if s == "in"}
    prefs = arg.preferred_extensions(af)
    for ext in prefs:
        # Theoreme : le noyau grounded survit dans CHAQUE monde maximal.
        assert grounded_in <= ext, (nom, grounded_in, ext)
    print(f"{nom:14s} grounded_in={sorted(grounded_in)} "
          f"preferred={[sorted(e) for e in prefs]}")

print("\nTemoin negatif OK : grounded <= chaque preferred sur les 4 instances.")

nixon 2        grounded_in=[] preferred=[[0], [1]]
tweety         grounded_in=[0, 2] preferred=[[0, 2]]
chaine a->b    grounded_in=[0] preferred=[[0]]
double defense grounded_in=[0, 2] preferred=[[0, 2]]

Temoin negatif OK : grounded <= chaque preferred sur les 4 instances.


Lecture du tableau : le noyau grandit avec la structure défensive (vide pour le Nixon pur — deux `undec` ; `{0, 2}` pour Tweety), et **chaque** préférée le contient. La réciproque est fausse en général — l'intersection des préférées peut **strictement** dépasser le grounded : c'est précisément l'objet du troisième exercice.

### Exercice 3 — quand l'intersection des préférées dépasse le grounded

**Contexte.** La section 6 a établi que le grounded est inclus dans chaque extension préférée, et a annoncé que la réciproque est fausse. **Objectif.** Construire un cadre de Dung à 4 arguments dont l'intersection des extensions préférées contient au moins un argument qui n'est pas `in` dans le labeling grounded. Mesurez-le avec `arg.DungAF`, `arg.grounded_labeling` et `arg.preferred_extensions`, puis calculez l'écart avec des ensembles Python.

**Indice.** Le grounded ne tranche rien sur deux attaquants qui s'excluent mutuellement. Si un argument `d` est attaqué par un argument `c` lui-même attaqué par ces deux attaquants, chaque extension préférée peut défendre `d` en choisissant **l'un** d'eux. Vérifiez que `d` appartient alors à toutes les préférées sans appartenir au grounded.

**Critère de vérification.** L'écart `intersection des préférées - ensemble in du grounded` est non vide, et son contenu est affiché avec les indices des arguments.


In [13]:
# Exercice 3 -- ecart grounded / sceptique
# TODO etudiant
def exercice_3_ecart_grounded_sceptique():
    print("Exercice 3 a completer")
    return None


exercice_3_ecart_grounded_sceptique()


Exercice 3 a completer


## À retenir

*Ce que le carnet a établi. (1) Les quantificateurs booléens se lisent stratégiquement — ordre de jeu, répondant intérieur — et l'ordre n'est pas commutatif (§2). (2) Un parseur à précédences sans parenthèses fait des choix de lecture qu'il faut savoir voir (§3). (3) L'acceptabilité crédule est un ∃ **matérialisé** par l'énumération des coalitions défensives — la chaîne défensive et le cycle impair sont les deux cas qui cassent l'intuition individuelle (§4). (4) L'acceptabilité sceptique est un ∀ sur les extensions préférées, calculées par l'organe natif Dung que ce carnet dote de `preferred_extensions` (§5). (5) Le théorème grounded ⊆ chaque préférée relie ce carnet au carnet Dung précédent — témoin négatif rejoué à chaque exécution (§6).*

## Vérifiez votre compréhension

Les trois exercices du carnet, chacun bornant une marche du quantificateur :

- **Exercice 1** — un ordre de quantificateurs qui change le verdict (§2).
- **Exercice 2** — prédire le témoin de l'énumération crédule (§4).
- **Exercice 3** — quand l'intersection des préférées dépasse le grounded (§6).

**Copie pédagogique déclarée** (règle `organ-first-implementation`) : `ict/qbf.py` porte le module EPITA `qbf_native.py` (tête `a5ac1a5d`, commit fondateur `0d2ac1b0`), divergences déclarées en docstring — l'acceptabilité sceptique consomme l'organe `ict.argumentation` au lieu de `dung_native` ; refus uniformisé `accepted=None` ; docstrings françaises. Fidélité prouvée par exécution différentielle (16/16 : `check_qbf`, `analyze_qbf`, `credulous_acceptance_qbf` byte-identiques) et 21 gates à littéraux figés. La fonction `preferred_extensions` est une **extension de l'organe natif** (pas une copie) : la série demande, l'organe calcule.

## Pour aller plus loin

**Références.** Dung, P. M. (1995). *On the Acceptability of Arguments and its Fundamental Role in Nonmonotonic Reasoning, Logic Programming, and n-Person Games*. Artificial Intelligence 77. · Papadimitriou, C. (1994). *Computational Complexity* — QBF et la hiérarchie polynomiale. · Pollock, J. (1987). *Defeasible Reasoning* — le Nixon diamond. · Source distillée : `2025-Epita-Intelligence-Symbolique`, `argumentation_analysis/agents/core/logic/qbf_native.py`.

- **[ICT-Argumentation-TruthMaintenance — la révision des croyances](ICT-Argumentation-TruthMaintenance-Python.ipynb)** : la couche révision du substrat argumentation — du JTMS à l'ATMS.
- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.

**Suivant — strate 6** : [→ ICT-Argumentation-TruthMaintenance-Python](ICT-Argumentation-TruthMaintenance-Python.ipynb) — la couche révision des croyances (JTMS/ATMS), troisième carnet de la strate avant le capstone.